In [8]:
import pandas as pd
import numpy as np
import os

DATA_PATH = r"PAMAP2_Dataset\PAMAP2_Dataset\Protocol"

def load_pamap2(path):
    all_data = []

    for file in os.listdir(path):
        if file.endswith(".dat"):
            file_path = os.path.join(path, file)

            data = pd.read_csv(file_path, sep=' ', header=None)
            all_data.append(data)

    data = pd.concat(all_data, ignore_index=True)
    
    # Replace invalid values
    data.replace(-1e+08, np.nan, inplace=True)
    
    # Drop missing
    data.dropna(inplace=True)

    return data

data = load_pamap2(DATA_PATH)
print(data.shape)

(259803, 54)


In [9]:
def extract_features_labels(data):
    # Remove rows with activity = 0 (invalid)
    data = data[data[1] != 0]

    y = data[1].values
    
    # Drop timestamp + activity
    X = data.drop(columns=[0, 1]).values

    return X, y

X_raw, y_raw = extract_features_labels(data)

print(X_raw.shape, y_raw.shape)

(175498, 52) (175498,)


In [10]:
import numpy as np

def create_windows(X, y, window_size=128, step=64):

    Xs, ys = [], []

    for i in range(0, len(X) - window_size, step):

        Xs.append(X[i:i + window_size])
        ys.append(y[i + window_size - 1])  # label of last frame

    return np.array(Xs), np.array(ys)

In [11]:
X_raw, y_raw = extract_features_labels(data)

X_win, y_win = create_windows(X_raw, y_raw)

print(X_win.shape, y_win.shape)

(2741, 128, 52) (2741,)


In [12]:
from sklearn.preprocessing import LabelEncoder

y_win = LabelEncoder().fit_transform(y_win)

In [13]:
X=X_win
y=y_win

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

In [15]:
def downsample_naive(X, factor=2):
    return X[:, ::factor, :]

def downsample_mean(X, factor=2):
    N, T, F = X.shape
    T_new = T // factor
    
    X = X[:, :T_new*factor, :]
    X = X.reshape(N, T_new, factor, F)
    
    return X.mean(axis=2)

In [16]:
from scipy.signal import butter, filtfilt, savgol_filter

def butter_filter(X, cutoff=5, fs=100, order=3):
    b, a = butter(order, cutoff / (0.5 * fs), btype='low')
    X_filtered = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = filtfilt(b, a, X[i, :, j])
    
    return X_filtered


def savgol_filtering(X, window=7, poly=2):
    X_filtered = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = savgol_filter(X[i, :, j], window, poly)
    
    return X_filtered

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, MaxPooling1D
from tensorflow.keras import Input

def build_tiny_cnn(input_shape, num_classes):

    model = Sequential([
        Input(shape=input_shape),
        Conv1D(16, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),

        Flatten(),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def get_models():
    return {
        "LogisticRegression": LogisticRegression(max_iter=200),
        "DecisionTree": DecisionTreeClassifier(max_depth=15, min_samples_split=10),
        "TinyCNN": "cnn"
    }

In [19]:
import time
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.base import clone

def train_and_evaluate(X_train, y_train, X_test, y_test, model):

    input_shape = X_train.shape

    # =====================================================
    # CNN CASE
    # =====================================================
    if isinstance(model, str) and model == "cnn":

        if len(X_train.shape) != 3:
            raise ValueError("CNN requires (samples, timesteps, features)")

        num_classes = len(np.unique(y_train))

        cnn = build_tiny_cnn(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            num_classes=num_classes
        )

        start_train = time.time()
        cnn.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)
        train_time = time.time() - start_train

        start_inf = time.time()
        y_pred = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
        inference_time = time.time() - start_inf

    # =====================================================
    # SKLEARN MODELS
    # =====================================================
    else:

        model_copy = clone(model)

        # SAFE FLATTENING (UCI-HAR compatible)
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_test_flat = X_test.reshape(X_test.shape[0], -1)

        start_train = time.time()
        model_copy.fit(X_train_flat, y_train)
        train_time = time.time() - start_train

        start_inf = time.time()
        y_pred = model_copy.predict(X_test_flat)
        inference_time = time.time() - start_inf

    # =====================================================
    # METRICS
    # =====================================================
    acc = accuracy_score(y_test, y_pred)
    inf_per_sample = inference_time / len(X_test)

    return {
        "accuracy": acc,
        "train_time": train_time,
        "inference_time": inference_time,
        "inference_time/sample": inf_per_sample,
        "input_shape": input_shape
    }

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
def run_all_experiments(X, y):

    results = []
    models = get_models()

    from sklearn.model_selection import train_test_split
    from sklearn.base import clone

    # Precompute
    X_naive = downsample_naive(X)
    X_mean = downsample_mean(X)
    X_butter = butter_filter(X)
    X_savgol = savgol_filtering(X)

    pipelines = [
        ("Raw", X, "No Processing"),

        ("Naive Downsample", X_naive, "Downsampling Only"),
        ("Mean Downsample", X_mean, "Downsampling Only"),

        ("Butterworth", X_butter, "Filtering Only"),
        ("Savitzky-Golay", X_savgol, "Filtering Only"),

        ("Naive + Butterworth", butter_filter(X_naive), "Downsampling → Filtering"),
        ("Mean + Butterworth", butter_filter(X_mean), "Downsampling → Filtering"),
        ("Naive + Savgol", savgol_filtering(X_naive), "Downsampling → Filtering"),
        ("Mean + Savgol", savgol_filtering(X_mean), "Downsampling → Filtering"),

        ("Butterworth → Naive", downsample_naive(X_butter), "Filtering → Downsampling"),
        ("Butterworth → Mean", downsample_mean(X_butter), "Filtering → Downsampling"),
        ("Savgol → Naive", downsample_naive(X_savgol), "Filtering → Downsampling"),
        ("Savgol → Mean", downsample_mean(X_savgol), "Filtering → Downsampling"),
    ]

    for model_name, model in models.items():

        print(f"\nRunning for model: {model_name}")

        for name, X_proc, pipeline_type in pipelines:

            X_train, X_test, y_train, y_test = train_test_split(
                X_proc, y, test_size=0.2, random_state=42
            )

            if model == "cnn":
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model)
            else:
                from sklearn.base import clone
                model_copy = clone(model)
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model_copy)

            metrics["name"] = name
            metrics["pipeline_type"] = pipeline_type
            metrics["model"] = model_name

            results.append(metrics)

    return pd.DataFrame(results)

In [22]:
def compute_score(row):
    return (
        row["accuracy"]
        - 0.1 * row["inference_time"]
        - 0.000001 * row["train_time"]
    )

In [23]:
df_results = run_all_experiments(X, y)

# df_results = df_results[
#     ["name", "pipeline_type", "accuracy", "train_time",
#      "inference_time", "inference_time_per_sample", "input_shape"]
# ]

# # 🔹 Sort (by accuracy, highest first)
# df_results = df_results.sort_values(by="accuracy", ascending=False)

# # 🔹 Reset index so it becomes clean 0,1,2,...
# df_results = df_results.reset_index(drop=True)

# print(df_results.to_string())


Running for model: LogisticRegression

Running for model: DecisionTree

Running for model: TinyCNN


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(sha

In [24]:
df_results["score"] = df_results.apply(compute_score, axis=1)

In [25]:
# models = get_models()

# models = df_results["model"].unique()

# for m in models:

#     print("\n" + "="*60)
#     print(f"🔹 MODEL: {m}")
#     print("="*60)

#     df_model = df_results[df_results["model"] == m]

#     # already sorted above, but ensure safety
#     df_model = df_model.sort_values(by="score", ascending=False)

#     # reorder columns EXACTLY like your target format
#     df_model = df_model[
#         ["model", "name", "pipeline_type",
#          "accuracy", "train_time",
#          "inference_time", "inference_time/sample",
#          "input_shape", "score"]
#     ]

#     print(df_model.to_string(index=False))

In [26]:
def format_results(df):
    cols = [
        "model",
        "name",
        "pipeline_type",
        "accuracy",
        "train_time",
        "inference_time",
        "inference_time/sample",
        "input_shape",
        "score"
    ]
    
    df = df[cols]
    
    # rounding for readability
    df["accuracy"] = df["accuracy"].round(6)
    df["train_time"] = df["train_time"].round(6)
    df["inference_time"] = df["inference_time"].round(6)
    df["inference_time/sample"] = df["inference_time/sample"].apply(lambda x: f"{x:.2e}")
    df["score"] = df["score"].round(6)
    
    return df

In [32]:
def print_model_results(df, model_name):
    print("\n" + "="*60)
    print(f"🔹 MODEL: {model_name}")
    print("="*60)
    
    model_df = df[df["model"] == model_name]
    # sort by score (descending = best first)
    model_df = model_df.sort_values(by="score", ascending=False)
    model_df = format_results(model_df)
    
    print(model_df.to_string(index=False))

In [33]:
for model in df_results["model"].unique():
    print_model_results(df_results, model)


🔹 MODEL: LogisticRegression
             model                name            pipeline_type  accuracy  train_time  inference_time inference_time/sample     input_shape    score
LogisticRegression Butterworth → Naive Filtering → Downsampling  0.932605    0.960896        0.003287              5.99e-06  (2192, 64, 52) 0.932275
LogisticRegression  Mean + Butterworth Downsampling → Filtering  0.932605    1.204613        0.003290              5.99e-06  (2192, 64, 52) 0.932274
LogisticRegression       Mean + Savgol Downsampling → Filtering  0.930783    1.043459        0.002818              5.13e-06  (2192, 64, 52) 0.930500
LogisticRegression Naive + Butterworth Downsampling → Filtering  0.928962    1.113596        0.003996              7.28e-06  (2192, 64, 52) 0.928561
LogisticRegression         Butterworth           Filtering Only  0.928962    1.704836        0.006027              1.10e-05 (2192, 128, 52) 0.928357
LogisticRegression  Butterworth → Mean Filtering → Downsampling  0.927140    